In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Exploratory Data Analysis

In [ ]:

df = pd.read_csv("/kaggle/input/competitions/playground-series-s6e3/train.csv")
print("Shape:", df.shape)
df.head()


In [ ]:
df['Churn'] = df['Churn'].map({'Yes':1, 'No':0})
df = df.drop(columns=['id'])
column_list = df.columns.tolist()
print(column_list)


In [ ]:
binary_cols = [col for col in df.columns if df[col].nunique() == 2 and col != 'Churn']
print(binary_cols)
print(df[binary_cols].dtypes)
for col in binary_cols:
    print(col, df[col].unique())

In [ ]:
binary_map = {
    'Yes':1,
    'No':0,
    'Male':1,
    'Female':0
}

for col in binary_cols:
    df[col] = df[col].replace(binary_map)

df[binary_cols].head()

In [ ]:
for col in binary_cols:
    print("\n", col)
    print(pd.crosstab(df[col], df['Churn'], normalize='index'))

In [ ]:
def churn_rate_by_feature(col):
    table = pd.crosstab(df[col], df['Churn'], normalize='index')
    print("\n", col)
    print(table)

for col in ['Contract','InternetService','PaymentMethod']:
    churn_rate_by_feature(col)

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

sns.boxplot(x='Churn', y='tenure', data=df)
plt.show()

sns.boxplot(x='Churn', y='MonthlyCharges', data=df)
plt.show()

sns.boxplot(x='Churn', y='TotalCharges', data=df)
plt.show()

In [ ]:
df.head()

In [ ]:
df_charges = df[['tenure', 'TotalCharges', 'MonthlyCharges', 'Churn']]
df_charges.head(50)

In [ ]:
df_charges[df_charges['Churn'] == "Yes"].head(50)

In [ ]:
pd.crosstab(
    [df['Contract'], df['InternetService']],
    df['Churn'],
    normalize='index'
)


In [ ]:
pd.crosstab(
    [df['Contract'], df['PaymentMethod']],
    df['Churn'],
    normalize='index'
)

In [ ]:
df['tenure_group'] = pd.cut(
    df['tenure'],
    bins=[0,12,24,48,72],
    labels=['0-1yr','1-2yr','2-4yr','4-6yr']
)
pd.crosstab(
    [df['tenure_group'], df['Contract']],
    df['Churn'],
    normalize='index'
)

In [ ]:
df['HasFamily'] = ((df['Partner'] == 1) | (df['Dependents'] == 1)).astype(int)
services = [
    'OnlineSecurity','OnlineBackup','DeviceProtection',
    'TechSupport','StreamingTV','StreamingMovies'
]

df['ServiceCount'] = df[services].apply(lambda x: (x=='Yes').sum(), axis=1)

df['ChargeRatio'] = df['TotalCharges'] / (df['tenure'] + 1)

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

plt.figure(figsize=(12,8))
sns.heatmap(df.corr(numeric_only=True), annot=True, cmap="coolwarm")
plt.show()

# MODEL DEVELOPMENT

In [ ]:
# Separate target
y = df['Churn']
X = df.drop(columns=['Churn'])

# One-hot encode categorical columns
X = pd.get_dummies(X, drop_first=True)
print(X.shape)

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [ ]:
print(y_test.head(10))
print(y_train.head(10))

In [ ]:
# Force correct, clean target – run this once right after splitting
y_train = y_train.astype('float32')
y_test  = y_test.astype('float32')
if y_train.hasnans or y_test.hasnans:
    print("NaN count in y_train:", y_train.isna().sum())
    print("NaN count in y_test :", y_test.isna().sum())
    # Optional: drop bad rows (but better to find root cause)
    # valid = y_train.notna() & X_train.index.isin(y_train.index)
    # X_train = X_train[valid]
    # y_train = y_train[valid]
    raise ValueError("Target contains NaN – fix before proceeding")

# Safety net: fail loudly if something is wrong
if y_train.isna().any() or y_test.isna().any():
    raise ValueError("NaN detected in target after cleaning – check earlier cells!")

print("After safety conversion:")
print(y_train.value_counts(dropna=False))
print(y_test.value_counts(dropna=False))

# Benchmark - Logistic Regression

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, roc_auc_score, classification_report

log_model = LogisticRegression(max_iter=2000)

log_model.fit(X_train, y_train)

y_pred = log_model.predict(X_test)
y_prob = log_model.predict_proba(X_test)[:,1]

print("Accuracy:", accuracy_score(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_prob))
print(classification_report(y_test, y_pred))

In [ ]:
import pandas as pd

coefficients = pd.DataFrame({
    "Feature": X.columns,
    "Importance": log_model.coef_[0]
})

coefficients.sort_values(by="Importance", ascending=False).head(10)

# Benchmark - Decision Tree

In [ ]:
from sklearn.tree import DecisionTreeClassifier

tree_model = DecisionTreeClassifier(
    max_depth=5,
    random_state=42
)

tree_model.fit(X_train, y_train)

y_pred = tree_model.predict(X_test)
y_prob = tree_model.predict_proba(X_test)[:,1]

print("ROC-AUC:", roc_auc_score(y_test, y_prob))
print(classification_report(y_test, y_pred))

# Benchmark | Random Forest | ROC-AUC: 91.246

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(
    n_estimators=300,
    max_depth=10,
    random_state=42
)

rf_model.fit(X_train, y_train)

y_pred = rf_model.predict(X_test)
y_prob = rf_model.predict_proba(X_test)[:,1]

print("ROC-AUC:", roc_auc_score(y_test, y_prob))
print(classification_report(y_test, y_pred))

In [ ]:
importances = pd.DataFrame({
    "Feature": X.columns,
    "Importance": rf_model.feature_importances_
})

importances.sort_values(by="Importance", ascending=False).head(15)

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

# -----------------------------
# 1️⃣ Prepare Data
# -----------------------------
# Convert Yes/No to 1/0
y_train = y_train.map({'No':0,'Yes':1})
y_test = y_test.map({'No':0,'Yes':1})
# Convert pandas to torch tensors
X_train_tensor = torch.from_numpy(X_train.values.astype(np.float32))
y_train_tensor = torch.tensor(y_train.values, dtype=torch.float32).view(-1,1)

X_test_tensor  = torch.from_numpy(X_test.values.astype(np.float32))
y_test_tensor = torch.tensor(y_test.values, dtype=torch.float32).view(-1,1)

# Create DataLoader for batching
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
test_dataset = TensorDataset(X_test_tensor, y_test_tensor)

train_loader = DataLoader(train_dataset, batch_size=512, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=512, shuffle=False)

In [ ]:
print("Type of y_train:", type(y_train))
print("Shape of y_train:", y_train.shape if hasattr(y_train, 'shape') else "no shape")

# Show first 10 values exactly as they are
print("\nFirst 10 raw values in y_train:")
print(y_train.head(10))

print("\nColumn names if it's a DataFrame:")
if isinstance(y_train, pd.DataFrame):
    print(y_train.columns.tolist())

In [ ]:
print("=== TARGET DIAGNOSTICS ===")
print("y_train dtype:", y_train.dtype)
print("y_train unique values:\n", y_train.value_counts(dropna=False))
print("y_train has NaN:", y_train.isna().any())
print("Number of NaN in y_train:", y_train.isna().sum())

print("\ny_test unique values:\n", y_test.value_counts(dropna=False))
print("y_test has NaN:", y_test.isna().any())
print("Number of NaN in y_test:", y_test.isna().sum())

# If you still have the original string version (before mapping), check it:
# print(y_train_original.value_counts(dropna=False))

# Neural Network | ROC-AUC 91.32

In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, roc_auc_score, classification_report, confusion_matrix

# ────────────────────────────────────────────────
# 1. Assume your data is already split
#    X_train, X_test, y_train, y_test are pandas objects
# ────────────────────────────────────────────────

# A. Make sure target is numeric (0/1)
if y_train.dtype == 'object':
    y_train = y_train.map({'No': 0, 'Yes': 1}).astype(np.float32)
if y_test.dtype == 'object':
    y_test = y_test.map({'No': 0, 'Yes': 1}).astype(np.float32)

# B. Identify columns that should be scaled (continuous features)
numeric_cols = ['tenure', 'MonthlyCharges', 'TotalCharges']
# Add any other continuous columns you have

# C. Scale numeric features
scaler = StandardScaler()

X_train[numeric_cols] = scaler.fit_transform(X_train[numeric_cols])
X_test[numeric_cols]  = scaler.transform(X_test[numeric_cols])

# D. Force ALL columns to float32 — this fixes bool → object dtype problem
X_train = X_train.astype(np.float32)
X_test  = X_test.astype(np.float32)

# Quick safety prints (you can comment out later)
print("X_train dtypes after conversion:\n", X_train.dtypes.value_counts())
print("X_train.values dtype:", X_train.values.dtype)          # should be float32
print("Any NaN in X_train?", np.isnan(X_train.values).any())
print("Any inf in X_train?", np.isinf(X_train.values).any())

# ────────────────────────────────────────────────
# 2. Create PyTorch tensors
# ────────────────────────────────────────────────

X_train_tensor = torch.from_numpy(X_train.to_numpy()).float()
y_train_tensor = torch.from_numpy(y_train.to_numpy()).float().view(-1, 1)

X_test_tensor  = torch.from_numpy(X_test.to_numpy()).float()
y_test_tensor  = torch.from_numpy(y_test.to_numpy()).float().view(-1, 1)

# ────────────────────────────────────────────────
# 3. DataLoaders
# ────────────────────────────────────────────────

train_ds = TensorDataset(X_train_tensor, y_train_tensor)
test_ds  = TensorDataset(X_test_tensor, y_test_tensor)

train_loader = DataLoader(train_ds, batch_size=256, shuffle=True)
test_loader  = DataLoader(test_ds,  batch_size=512, shuffle=False)

# ────────────────────────────────────────────────
# 4. Model (NO final sigmoid — using BCEWithLogitsLoss)
# ────────────────────────────────────────────────

class ChurnModel(nn.Module):
    def __init__(self, input_size):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_size, 128),
            nn.ReLU(),
            nn.BatchNorm1d(128),
            nn.Dropout(0.3),
            
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.BatchNorm1d(64),
            nn.Dropout(0.25),
            
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Dropout(0.2),
            
            nn.Linear(32, 1)          # ← raw logits
        )
    
    def forward(self, x):
        return self.net(x)


model = ChurnModel(input_size=X_train.shape[1])
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

# ────────────────────────────────────────────────
# 5. Loss + Optimizer
# ────────────────────────────────────────────────

criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(), lr=0.0005, weight_decay=1e-5)

# ────────────────────────────────────────────────
# 6. Training loop with NaN protection
# ────────────────────────────────────────────────

epochs = 80
best_loss = float('inf')

for epoch in range(epochs):
    model.train()
    total_loss = 0.0
    batch_count = 0
    
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        
        optimizer.zero_grad()
        logits = model(xb)
        loss = criterion(logits, yb)
        
        if torch.isnan(loss) or torch.isinf(loss):
            print(f"NaN/Inf detected at epoch {epoch+1} — stopping")
            break
        
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=4.0)
        optimizer.step()
        
        total_loss += loss.item()
        batch_count += 1
    
    if batch_count == 0:
        print("Training collapsed — no valid batches")
        break
    
    avg_loss = total_loss / batch_count
    print(f"Epoch {epoch+1:2d}/{epochs}   loss: {avg_loss:.6f}")
    
    if avg_loss < best_loss:
        best_loss = avg_loss
        torch.save(model.state_dict(), "best_churn_model.pt")

print("\nTraining finished.\n")

# ────────────────────────────────────────────────
# 7. Evaluation
# ────────────────────────────────────────────────

model.eval()
all_probs = []
all_labels = []

with torch.no_grad():
    for xb, yb in test_loader:
        xb = xb.to(device)
        logits = model(xb)
        probs = torch.sigmoid(logits).cpu().numpy()
        all_probs.append(probs)
        all_labels.append(yb.numpy())

probs = np.concatenate(all_probs).ravel()
labels = np.concatenate(all_labels).ravel()

preds = (probs >= 0.5).astype(int)

print("Accuracy:      {:.4f}".format(accuracy_score(labels, preds)))
print("ROC AUC:       {:.4f}".format(roc_auc_score(labels, probs)))
print("\nClassification Report:")
print(classification_report(labels, preds, target_names=["Stay", "Churn"]))

print("\nConfusion Matrix:")
print(confusion_matrix(labels, preds))

# XGB-BOOST | ROC-AUC: 91.645

In [ ]:
from xgboost import XGBClassifier
from sklearn.metrics import roc_auc_score, classification_report, accuracy_score

xgb = XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric="auc",
    random_state=42
)

xgb.fit(X_train, y_train)

pred = xgb.predict(X_test)
prob = xgb.predict_proba(X_test)[:,1]

print("Accuracy:", accuracy_score(y_test, pred))
print("ROC-AUC:", roc_auc_score(y_test, prob))
print(classification_report(y_test, pred))

# LIGHT GBM | ROC-AUC: 91.658

In [ ]:
from lightgbm import LGBMClassifier

lgbm = LGBMClassifier(
    n_estimators=400,
    learning_rate=0.05,
    max_depth=-1,
    num_leaves=64,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)

lgbm.fit(X_train, y_train)

pred = lgbm.predict(X_test)
prob = lgbm.predict_proba(X_test)[:,1]

print("Accuracy:", accuracy_score(y_test, pred))
print("ROC-AUC:", roc_auc_score(y_test, prob))
print(classification_report(y_test, pred))

# CAT BOOST | ROC-AUC: 91.592

In [ ]:
from catboost import CatBoostClassifier

cat = CatBoostClassifier(
    iterations=400,
    learning_rate=0.05,
    depth=6,
    loss_function='Logloss',
    eval_metric='AUC',
    verbose=False
)

cat.fit(X_train, y_train)

pred = cat.predict(X_test)
prob = cat.predict_proba(X_test)[:,1]

print("Accuracy:", accuracy_score(y_test, pred))
print("ROC-AUC:", roc_auc_score(y_test, prob))
print(classification_report(y_test, pred))

# Prediction Test Data

In [138]:
X_test.head()

,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,PaperlessBilling,MonthlyCharges,TotalCharges,HasFamily,...,StreamingMovies_No internet service,StreamingMovies_Yes,Contract_One year,Contract_Two year,PaymentMethod_Credit card (automatic),PaymentMethod_Electronic check,PaymentMethod_Mailed check,tenure_group_1-2yr,tenure_group_2-4yr,tenure_group_4-6yr
493665,0,0,0,0,2,1,1,70.05,165.45,0,...,False,False,False,False,False,False,True,False,False,False
128315,0,0,1,1,42,1,0,20.65,849.85,1,...,True,False,False,True,False,False,True,False,True,False
430597,1,0,1,1,61,1,1,105.00,6077.75,1,...,False,True,False,True,False,True,False,False,False,True
341073,0,0,0,0,7,1,1,76.00,574.35,0,...,False,False,False,False,False,True,False,False,False,False
513192,1,0,1,0,59,1,0,89.90,5683.60,1,...,False,False,False,False,True,False,False,False,False,True


In [142]:
# ============================================
# Kaggle Churn Prediction Submission Script
# ============================================

import pandas as pd
import logging

# -----------------------------
# Logging Setup
# -----------------------------

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s"
)

logging.info("Starting Kaggle prediction pipeline")

# -----------------------------
# Load Test Dataset
# -----------------------------

TEST_PATH = "/kaggle/input/competitions/playground-series-s6e3/test.csv"

logging.info("Loading test dataset")

test_df = pd.read_csv(TEST_PATH)

logging.info(f"Test dataset shape: {test_df.shape}")

print(test_df.head())

# -----------------------------
# Separate ID column
# -----------------------------

logging.info("Separating ID column")

ids = test_df["id"]

# Remove id for model
X_test_kaggle = test_df.drop("id", axis=1)

logging.info(f"Feature dataset shape: {X_test_kaggle.shape}")

# -----------------------------
# Apply same preprocessing as training
# -----------------------------

logging.info("Encoding categorical features")

X_test_kaggle = pd.get_dummies(X_test_kaggle)

logging.info("Aligning test features with training features")

X_test_kaggle = X_test_kaggle.reindex(columns=X_train.columns, fill_value=0)

print("Final test feature shape:", X_test_kaggle.shape)

# -----------------------------
# Generate Predictions
# -----------------------------

logging.info("Generating churn probability predictions")

test_prob = lgbm.predict_proba(X_test_kaggle)[:, 1]

logging.info("Predictions generated successfully")

# -----------------------------
# Create Submission File
# -----------------------------

submission = pd.DataFrame({
    "id": ids,
    "Churn": test_prob
})

print("\nSubmission Preview\n")
print(submission.head())

# -----------------------------
# Save Submission
# -----------------------------

OUTPUT_FILE = "submission.csv"

submission.to_csv(OUTPUT_FILE, index=False)

logging.info(f"Submission file saved -> {OUTPUT_FILE}")

# -----------------------------
# Validation Checks
# -----------------------------

print("\nSubmission Shape:", submission.shape)

print("\nMissing Values:\n", submission.isnull().sum())

logging.info("Prediction pipeline completed successfully")

2026-03-10 07:39:06,269 | INFO | Starting Kaggle prediction pipeline
2026-03-10 07:39:06,270 | INFO | Loading test dataset
2026-03-10 07:39:07,031 | INFO | Test dataset shape: (254655, 20)
2026-03-10 07:39:07,037 | INFO | Separating ID column
2026-03-10 07:39:07,079 | INFO | Feature dataset shape: (254655, 19)
2026-03-10 07:39:07,080 | INFO | Encoding categorical features


       id  gender  SeniorCitizen Partner Dependents  tenure PhoneService  \
0  594194  Female              0     Yes         No      72          Yes   
1  594195  Female              0     Yes         No      71          Yes   
2  594196    Male              0      No         No      12          Yes   
3  594197    Male              0     Yes        Yes      71          Yes   
4  594198  Female              0      No         No      15          Yes   

  MultipleLines InternetService       OnlineSecurity         OnlineBackup  \
0           Yes     Fiber optic                  Yes                  Yes   
1            No              No  No internet service  No internet service   
2            No             DSL                  Yes                  Yes   
3           Yes             DSL                  Yes                   No   
4            No     Fiber optic                  Yes                   No   

      DeviceProtection          TechSupport          StreamingTV  \
0           

2026-03-10 07:39:07,367 | INFO | Aligning test features with training features
2026-03-10 07:39:07,375 | INFO | Generating churn probability predictions


Final test feature shape: (254655, 36)


2026-03-10 07:39:10,840 | INFO | Predictions generated successfully



Submission Preview

       id     Churn
0  594194  0.069357
1  594195  0.002529
2  594196  0.122942
3  594197  0.016611
4  594198  0.712019


2026-03-10 07:39:11,396 | INFO | Submission file saved -> submission.csv
2026-03-10 07:39:11,399 | INFO | Prediction pipeline completed successfully



Submission Shape: (254655, 2)

Missing Values:
 id       0
Churn    0
dtype: int64
